# 04 — Evaluation

AUC, F1, LogLoss, Brier score on the test set. Diagnostic plots.

**Requires:** Trained model, test_pairs from notebook 01.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, log_loss, accuracy_score,
    precision_recall_curve, average_precision_score,
    brier_score_loss, roc_curve
)
from sklearn.calibration import calibration_curve
import tensorflow as tf

## Prediction Loop

In [ ]:
# Uses resize_and_normalize from notebook 02

def predict_test_set(model, test_pairs, frame_count=12, frame_step=2):
    y_true, y_pred = [], []

    for i, (real_path, fake_path) in enumerate(test_pairs):
        if i % 20 == 0:
            print(f"  {i}/{len(test_pairs)}")

        real_raw, fake_raw = extract_frames(real_path, fake_path,
                                            frame_step=frame_step,
                                            frame_count=frame_count)

        # normalize (no augmentation)
        norm_real, norm_fake = [], []
        for r, f in zip(real_raw, fake_raw):
            r_norm, f_norm = resize_and_normalize(r, f, augment=False,
                                                   augment_instance=None)
            norm_real.append(r_norm)
            norm_fake.append(f_norm)

        real_input = np.expand_dims(np.array(norm_real), axis=0)
        fake_input = np.expand_dims(np.array(norm_fake), axis=0)

        real_score = float(model.predict(real_input, verbose=0).mean())
        fake_score = float(model.predict(fake_input, verbose=0).mean())

        y_true.extend([1, 0])
        y_pred.extend([real_score, fake_score])

    return np.array(y_true), np.array(y_pred)

y_true, y_pred = predict_test_set(model, test_pairs)
print(f"Predictions: {len(y_pred)}")

## Metrics

In [ ]:
def print_metrics(y_true, y_pred):
    labels = (y_pred > 0.5).astype(int)

    auc  = roc_auc_score(y_true, y_pred)
    f1   = f1_score(y_true, labels)
    ll   = log_loss(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    acc  = accuracy_score(y_true, labels)
    ap   = average_precision_score(y_true, y_pred)

    print(f"AUC:        {auc:.4f}")
    print(f"F1:         {f1:.4f}")
    print(f"LogLoss:    {ll:.4f}")
    print(f"Brier:      {brier:.4f}")
    print(f"Accuracy:   {acc:.4f}")
    print(f"Avg Prec:   {ap:.4f}")

print_metrics(y_true, y_pred)

## Diagnostic Plots

In [ ]:
def plot_diagnostics(y_true, y_pred):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    axes[0,0].plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    axes[0,0].plot([0,1],[0,1], "--", alpha=0.5)
    axes[0,0].set_title("ROC Curve"); axes[0,0].legend()
    axes[0,0].set_xlabel("FPR"); axes[0,0].set_ylabel("TPR")

    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    axes[0,1].step(rec, prec, where="post", label=f"AP={ap:.3f}")
    axes[0,1].set_title("Precision-Recall"); axes[0,1].legend()
    axes[0,1].set_xlabel("Recall"); axes[0,1].set_ylabel("Precision")

    # Calibration
    prob_true, prob_pred = calibration_curve(y_true, y_pred, n_bins=10)
    axes[1,0].plot(prob_pred, prob_true, "o-", label="Model")
    axes[1,0].plot([0,1],[0,1], "--", label="Perfect")
    axes[1,0].set_title("Calibration"); axes[1,0].legend()

    # Histogram
    axes[1,1].hist(y_pred, bins=30, edgecolor="black", alpha=0.7)
    axes[1,1].set_title("Prediction Distribution")
    axes[1,1].set_xlabel("Predicted Probability")

    for ax in axes.flat: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_diagnostics(y_true, y_pred)